# Módulo 03 · Aula 4 — O processo de Análise Exploratória de Dados

**Capacitação Introdutória de Ciência de Dados · FEA.dev**

---

Você já tem as ferramentas: pandas para manipular, gráficos para enxergar, estatística
para medir. Falta a parte que ninguém ensina explicitamente — **o que fazer com elas, e
em que ordem**.

Análise Exploratória de Dados (EDA) não é uma lista de comandos a executar. É um
**ciclo**: você faz uma pergunta, explora os dados atrás dela, descobre algo, formula uma
hipótese sobre o porquê — e essa hipótese vira a próxima pergunta.

Esta aula percorre esse ciclo quatro vezes, do começo ao fim, com dados reais do mercado
brasileiro. Preste atenção menos no código (você já conhece quase tudo) e mais no
**raciocínio entre as células**.

**Tempo estimado:** 90 minutos.

### Antes de começar — se você está no Google Colab

Este notebook lê arquivos da pasta `data/` do repositório, e no Colab a máquina começa vazia. **Execute a célula abaixo antes de qualquer outra**: ela traz o repositório e entra na pasta deste módulo, de modo que os caminhos `../data/...` usados no material funcionem sem alteração.

No VS Code ou no Jupyter local a célula não faz nada — os arquivos já estão no seu disco.

In [ ]:
# Setup do Google Colab.
# Traz o repositório da capacitação e entra na pasta deste módulo, para que os
# caminhos "../data/..." usados no material funcionem sem nenhuma alteração.
# Fora do Colab (VS Code, Jupyter local) esta célula não faz nada.
# Pode ser executada mais de uma vez sem problema.
import os
import subprocess
import sys

PASTA_DESTE_MODULO = "03_Visualizacao_EDA"
REPOSITORIO = "https://github.com/gustavokatsuo/Introducao-a-Ciencia-de-Dados.git"

if "google.colab" in sys.modules and not os.path.isdir("../data"):
    destino = "/content/Introducao-a-Ciencia-de-Dados"
    if not os.path.isdir(destino):
        print("Baixando o material da capacitação...")
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, destino], check=True)
    os.chdir(os.path.join(destino, PASTA_DESTE_MODULO))
    print("Pronto. Pasta de trabalho:", os.getcwd())

In [ ]:
from IPython.display import Image
Image("../assets/fluxo_eda.png", width=880)

## 1. O que é EDA — e o que não é

O termo foi cunhado por **John Tukey** nos anos 1970, num livro que defendia uma ideia
então controversa: antes de testar hipóteses formalmente, é preciso *olhar* para os
dados. Sem hipótese pré-definida, sem modelo, sem teste. Só olhar, com método.

| EDA **é** | EDA **não é** |
|---|---|
| Descobrir o que os dados têm a dizer | Confirmar o que você já queria concluir |
| Iterativo — cada resposta gera uma pergunta | Um roteiro linear executado uma vez |
| Documentado: cada decisão fica escrita | Uma sequência de células sem texto |
| Honesto sobre limitações | Uma seleção dos resultados bonitos |
| Descritivo: sobre o passado observado | Previsão sobre o futuro |

> **Atenção — O maior risco da EDA é você mesmo.** Explorando o suficiente, sempre aparece
> alguma coisa interessante — mesmo em dados aleatórios (vimos isso na aula anterior).
> A defesa é escrever a pergunta **antes** de olhar e registrar também o que **não**
> funcionou. Um notebook onde tudo deu certo é um notebook incompleto.

### As quatro etapas do ciclo

1. **PERGUNTA** — o que eu quero descobrir? Específica o bastante para ser respondida.
2. **EXPLORAÇÃO** — que recorte, que resumo, que gráfico responde a isso?
3. **DESCOBERTA** — o que apareceu? Inclusive o que contraria o esperado.
4. **HIPÓTESE** — por que isso acontece? O que precisaria ser verdade?

E aí a hipótese vira a pergunta seguinte.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 130)

## 2. Etapa 0 — conhecer os dados antes de perguntar

Antes de qualquer pergunta, é preciso saber **o que é cada coluna, de onde veio e o que
falta**. Pular esta etapa é a origem da maioria das análises erradas.

In [ ]:
acoes = pd.read_csv("../data/acoes_b3.csv", parse_dates=["data"])
empresas = pd.read_csv("../data/empresas_b3.csv")
indicadores = pd.read_csv("../data/indicadores_macro.csv", parse_dates=["data"])

print("acoes      :", acoes.shape)
print("empresas   :", empresas.shape)
print("indicadores:", indicadores.shape)
acoes.head(3)

In [ ]:
acoes.info()

In [ ]:
print("Período  :", acoes["data"].min().date(), "->", acoes["data"].max().date())
print("Ativos   :", sorted(acoes["ticker"].unique()))
print("Faltantes:", acoes.isna().sum().sum())
print()
print("Pregões por ativo:")
print(acoes["ticker"].value_counts())

Oito ativos, cinco anos, sem valores faltantes, e **todos com o mesmo número de
pregões** — 1.246. Isso é uma boa notícia: as séries estão alinhadas no tempo, o que
permite compará-las diretamente.

### O que sabemos sobre a origem

- **Fonte:** Yahoo Finance (preços) e Banco Central (indicadores macro). Estão
  vendorizados em `data/`, com a coleta documentada em `data/coleta/coletar_dados.py`.
- **`fechamento`** é o preço de fecho do pregão, ajustado por desdobramentos.
- **`fechamento_ajustado`** ajusta também por dividendos — é o que mede o retorno de
  quem investiu, e é o que devemos usar.

### E o que já sabemos que está errado

Três limitações que precisam ser declaradas **agora**, e não escondidas no fim:

1. **Viés de sobrevivência.** Estes oito papéis foram escolhidos por serem líquidos
   *hoje*. Empresas que quebraram ou saíram da bolsa no período não estão aqui. Qualquer
   retorno médio que calcularmos é, por construção, otimista.
2. **Amostra pequena e enviesada.** Oito ações não representam a B3 (que tem centenas), e
   são todas grandes empresas.
3. **Período particular.** 2021–2025 inclui pós-pandemia, um ciclo agressivo de alta de
   juros e duas eleições. Não é um período "típico" — nenhum é.

> Anote as limitações **antes** de começar. Depois de encontrar um resultado bonito, a
> tentação de minimizá-las é grande demais.

In [ ]:
# Enriquecendo com o cadastro e calculando o retorno diário por ativo
acoes = acoes.merge(empresas, on="ticker", how="left", validate="m:1")
acoes = acoes.sort_values(["ticker", "data"])
acoes["retorno_diario"] = acoes.groupby("ticker")["fechamento_ajustado"].pct_change() * 100
acoes["ano"] = acoes["data"].dt.year
acoes["ano_mes"] = acoes["data"].dt.to_period("M")

print("Base pronta:", acoes.shape)
acoes[["data", "ticker", "empresa", "setor", "fechamento_ajustado", "retorno_diario"]].head(3)

---

## 3. Ciclo 1 — Quem ganhou e quem perdeu?

### PERGUNTA

*Qual foi o retorno acumulado de cada ativo entre 2021 e 2025?*

Começamos pela pergunta mais simples possível. É de propósito: a primeira volta do ciclo
serve para entender o terreno, não para descobrir algo sofisticado.

In [ ]:
# EXPLORAÇÃO
# Ordenamos por data ANTES de agrupar, para que "first" e "last" signifiquem
# de fato o primeiro e o último pregão de cada ativo.
extremos = (
    acoes.sort_values("data")
    .groupby("ticker")["fechamento_ajustado"]
    .agg(primeiro="first", ultimo="last")
)
extremos["retorno_acumulado_%"] = (extremos["ultimo"] / extremos["primeiro"] - 1) * 100

desempenho = (
    extremos.reset_index()
    .merge(empresas[["ticker", "empresa", "setor"]], on="ticker")
    .sort_values("retorno_acumulado_%", ascending=False)
)

desempenho.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

cores = ["#c0392b" if valor < 0 else "#27ae60"
         for valor in desempenho["retorno_acumulado_%"]]
ax.barh(desempenho["ticker"], desempenho["retorno_acumulado_%"], color=cores)

ax.axvline(0, color="black", linewidth=0.9)
ax.set_title("Retorno acumulado por ativo — 2021 a 2025", fontsize=13, pad=12)
ax.set_xlabel("Retorno acumulado (%)")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)

plt.show()

### DESCOBERTA

Duas coisas saltam:

1. **A dispersão é enorme.** Entre o melhor e o pior ativo há uma diferença de centenas
   de pontos percentuais. "O mercado" não teve um desempenho — cada papel teve o seu.
2. **MGLU3 destoa violentamente para baixo**, com perda superior a 90%. Não é uma queda
   ruim: é uma quase destruição do capital investido.

### HIPÓTESE

Se os retornos são tão diferentes entre si, algo os separa. Três candidatos plausíveis:

- **H1 — risco:** ativos mais arriscados entregaram mais retorno (a teoria diz que
  deveria ser assim);
- **H2 — setor:** setores diferentes reagiram de forma diferente ao período;
- **H3 — juros:** o ciclo de alta da Selic castigou empresas dependentes de crédito e
  consumo.

Cada hipótese é a pergunta do próximo ciclo.

---

## 4. Ciclo 2 — O risco explica o retorno?

### PERGUNTA

*Ativos mais voláteis entregaram retornos maiores no período?*

In [ ]:
# EXPLORAÇÃO
risco_retorno = acoes.groupby("ticker")["retorno_diario"].agg(
    retorno_medio="mean", volatilidade="std"
)
risco_retorno["retorno_anual"] = risco_retorno["retorno_medio"] * 252
risco_retorno["risco_anual"] = risco_retorno["volatilidade"] * np.sqrt(252)

correlacao = risco_retorno["risco_anual"].corr(risco_retorno["retorno_anual"])
print(f"Correlação entre risco e retorno (8 ativos): {correlacao:.3f}")

risco_retorno[["retorno_anual", "risco_anual"]].round(1).sort_values("risco_anual")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(risco_retorno["risco_anual"], risco_retorno["retorno_anual"],
           s=150, color="#1f4e79", zorder=3)
for ticker, linha in risco_retorno.iterrows():
    ax.annotate(ticker, (linha["risco_anual"], linha["retorno_anual"]),
                xytext=(9, 5), textcoords="offset points", fontsize=11)

ax.axhline(0, color="black", linewidth=0.9)
ax.set_title("Risco × retorno anualizados — 2021 a 2025", fontsize=13, pad=12)
ax.set_xlabel("Volatilidade anualizada (%)")
ax.set_ylabel("Retorno anualizado (%)")
ax.grid(alpha=0.3)

plt.show()

### DESCOBERTA

A correlação é **negativa**: no período, mais risco veio acompanhado de *menos* retorno —
o oposto do que a teoria de finanças prevê no longo prazo.

Mas olhe o gráfico antes de acreditar no número. A correlação está sendo dominada por
**um único ponto**: MGLU3, no canto inferior direito, com o maior risco e o pior retorno.
Com apenas 8 observações, um ponto extremo manda no coeficiente inteiro.

Vamos testar isso diretamente — remover uma observação e ver o que sobra do resultado é
um teste de robustez elementar e sempre revelador.

In [ ]:
sem_mglu = risco_retorno.drop("MGLU3")
correlacao_sem = sem_mglu["risco_anual"].corr(sem_mglu["retorno_anual"])

print(f"Com MGLU3 (n=8): {correlacao:.3f}")
print(f"Sem MGLU3 (n=7): {correlacao_sem:.3f}")

A correlação praticamente desaparece. **A "descoberta" era um artefato de um ponto.**

> **Atenção:** Isto **não** significa que devemos remover MGLU3. Significa que a conclusão
> "risco não paga" não se sustenta: ela dependia inteiramente de um caso. Com 8 ativos e
> 5 anos, simplesmente **não há dados suficientes** para responder a essa pergunta — que,
> aliás, é objeto de uma literatura acadêmica de décadas, feita com milhares de ações e
> muitas décadas de histórico.

### HIPÓTESE

A resposta honesta ao ciclo 2 é: *não dá para responder com esta amostra.* Registramos
isso e seguimos — mas ficou uma pergunta muito mais interessante no caminho: **o que
aconteteu com a MGLU3?**

---

## 5. Ciclo 3 — O que aconteceu com a MGLU3?

### PERGUNTA

*A queda da MGLU3 foi um evento pontual ou um processo contínuo? E ela coincide com
alguma coisa observável nos dados macroeconômicos?*

In [ ]:
# EXPLORAÇÃO — primeiro, a forma da queda
fig, ax = plt.subplots(figsize=(12, 5))

for ticker in acoes["ticker"].unique():
    serie = acoes[acoes["ticker"] == ticker].sort_values("data")
    base = serie["fechamento_ajustado"].iloc[0]
    destaque = ticker == "MGLU3"
    ax.plot(serie["data"], serie["fechamento_ajustado"] / base * 100,
            linewidth=2.2 if destaque else 1.0,
            color="#c0392b" if destaque else "#b0b0b0",
            label=ticker if destaque else None,
            zorder=3 if destaque else 1)

ax.axhline(100, color="black", linestyle="--", linewidth=1)
ax.set_title("Retorno acumulado, base 100 — MGLU3 em destaque", fontsize=13, pad=12)
ax.set_xlabel("Data")
ax.set_ylabel("Índice (100 = 04/01/2021)")
ax.legend()
ax.grid(alpha=0.3)

plt.show()

A queda não foi um evento isolado: foi um **processo**, espalhado por vários anos. Vamos
olhar ano a ano, porque o gráfico em escala comprimida esconde o que aconteceu depois de
2022.

In [ ]:
mglu = acoes[acoes["ticker"] == "MGLU3"].sort_values("data")

por_ano = mglu.groupby("ano")["fechamento_ajustado"].agg(["first", "last"])
por_ano["retorno_%"] = ((por_ano["last"] / por_ano["first"] - 1) * 100).round(1)
por_ano.round(2)

A tabela corrige a impressão do gráfico: houve quedas violentas em **2021 (−71%)** e
**2022 (−59%)**, uma queda menor em 2023, **outra queda de 66% em 2024** e uma
recuperação parcial em 2025. Não foi um tombo seguido de estabilização — foram vários
tombos.

> **Lição de método:** um gráfico em escala comprimida faz tudo que acontece perto do
> zero parecer "estável". Depois de cair 95%, mais uma queda de 66% é quase invisível no
> desenho — e é enorme para quem estava investido. Sempre confira a impressão visual com
> os números.

Agora o cruzamento com os indicadores macro.

In [ ]:
# Retorno mensal de cada ativo
mensal = (
    acoes.groupby(["ticker", "ano_mes"])["fechamento_ajustado"]
    .agg(["first", "last"])
    .reset_index()
)
mensal["retorno_mes"] = (mensal["last"] / mensal["first"] - 1) * 100

# Indicadores macro no mesmo período mensal
indicadores["ano_mes"] = indicadores["data"].dt.to_period("M")
painel = mensal.merge(
    indicadores[["ano_mes", "ipca_mes_pct", "selic_mes_pct", "dolar_medio"]],
    on="ano_mes", how="left", validate="m:1",
)

print("Observações por ativo:", painel.groupby("ticker").size().unique())
painel.head(3)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

selic = indicadores.sort_values("data")
ax.plot(selic["data"], selic["selic_mes_pct"], color="#1f4e79", linewidth=2)
ax.fill_between(selic["data"], 0, selic["selic_mes_pct"], color="#1f4e79", alpha=0.12)

ax.set_title("Selic acumulada no mês (2021–2025)", fontsize=13, pad=12)
ax.set_xlabel("Data")
ax.set_ylabel("% ao mês")
ax.grid(alpha=0.3)

plt.show()

Aí está o contexto: a Selic saiu de perto de 0,15% ao mês (2% ao ano) no início de 2021
e subiu de forma acelerada ao longo de 2021 e 2022 — exatamente o período da queda da
MGLU3.

Isso é **coincidência temporal**, não causalidade. Mas é uma coincidência com mecanismo
plausível, e mecanismo é o que autoriza investigar: varejo depende de crédito ao
consumidor, e empresas em crescimento são avaliadas por fluxos de caixa futuros, cujo
valor presente encolhe quando os juros sobem.

Vamos medir a associação entre retorno mensal e Selic para cada ativo.

In [ ]:
# Percorremos os grupos com um for e montamos a tabela linha a linha.
# É mais verboso que um .apply(), e bem mais fácil de ler e depurar.
linhas = []
for ticker, grupo in painel.groupby("ticker"):
    linhas.append({
        "ticker": ticker,
        "corr_selic": grupo["retorno_mes"].corr(grupo["selic_mes_pct"]),
        "corr_ipca": grupo["retorno_mes"].corr(grupo["ipca_mes_pct"]),
        "corr_dolar": grupo["retorno_mes"].corr(grupo["dolar_medio"]),
        "meses": grupo["retorno_mes"].notna().sum(),
    })

correlacoes = pd.DataFrame(linhas).set_index("ticker").round(3)
correlacoes.sort_values("corr_selic")

### DESCOBERTA

**A hipótese não apareceu nos dados.** Duas coisas contrariam o que esperávamos:

1. Todas as correlações com a Selic são **próximas de zero** — nenhuma passa de 0,22 em
   módulo, e com 60 observações mensais valores dessa ordem não se distinguem de zero com
   qualquer confiança;
2. MGLU3 **não é** o ativo com a correlação mais negativa. Nem sequer é negativa: é
   ligeiramente positiva.

Pare um segundo aqui, porque este é o momento mais instrutivo da aula. Tínhamos uma
história coerente, com mecanismo econômico plausível e coincidência temporal evidente no
gráfico — e a medição não a sustenta.

O que fazer? **Não é reescrever a pergunta até o número obedecer.** É entender por que a
medição não capta o que o gráfico sugere. Duas razões, e a primeira é um erro nosso:

- **O cruzamento está conceitualmente errado.** Correlacionamos o retorno mensal com o
  **nível** da Selic naquele mês. Preços reagem a *surpresas*, não a níveis já
  conhecidos: se o mercado inteiro já sabe que a Selic está em 13%, isso está no preço
  desde o dia em que passou a ser esperado. O certo seria usar a **variação** da taxa —
  ou, melhor ainda, a diferença em relação ao que se esperava.
- **A coincidência temporal não é evidência estatística.** Duas séries que se movem
  juntas ao longo de um período longo produzem um gráfico convincente e uma correlação
  fraca, porque a correlação mensal mede co-movimento **mês a mês**, não tendências de
  vários anos.

Vamos corrigir o primeiro ponto e ver se muda alguma coisa.

In [ ]:
# Refazendo com a VARIAÇÃO da Selic, que é o que carrega surpresa
indicadores_ordenado = indicadores.sort_values("ano_mes").copy()
indicadores_ordenado["variacao_selic"] = indicadores_ordenado["selic_mes_pct"].diff()

painel = painel.merge(
    indicadores_ordenado[["ano_mes", "variacao_selic"]], on="ano_mes", how="left"
)

linhas = []
for ticker, grupo in painel.groupby("ticker"):
    linhas.append({
        "ticker": ticker,
        "corr_nivel_selic": grupo["retorno_mes"].corr(grupo["selic_mes_pct"]),
        "corr_variacao_selic": grupo["retorno_mes"].corr(grupo["variacao_selic"]),
    })

comparacao = (
    pd.DataFrame(linhas).set_index("ticker").round(3)
    .sort_values("corr_variacao_selic")
)

comparacao

Com a variação, as correlações mudam de sinal para vários ativos e passam a ser
majoritariamente negativas — a direção que a teoria prevê. Mas continuam pequenas, e
MGLU3 segue sem se destacar: quem tem a correlação mais negativa é WEGE3.

### HIPÓTESE

A conclusão honesta deste ciclo:

> *"A queda da MGLU3 coincide temporalmente com o ciclo de alta de juros e existe um
> mecanismo econômico plausível ligando os dois. **Não encontramos evidência disso nos
> nossos dados**: as correlações entre retorno mensal e Selic são indistinguíveis de zero,
> e MGLU3 não se separa dos demais ativos por esse critério. Com 60 observações mensais e
> uma única empresa, não temos como separar um eventual efeito de juros de tudo o mais
> que aconteceu com ela no período — concorrência, execução, endividamento e a
> normalização do e-commerce depois da pandemia."*

O que faríamos a seguir: incluir dezenas de varejistas para poder comparar, usar variação
**não antecipada** de juros, controlar por endividamento e olhar as demonstrações
financeiras da empresa. Isso já é outro projeto — e é exatamente o passo natural depois
desta capacitação.

> **Atenção — Um ciclo que "não deu em nada" não é um ciclo perdido.** Você agora sabe que
> a explicação fácil não se sustenta com esses dados, sabe *por que* a medição inicial
> estava errada, e sabe o que seria preciso para responder de verdade. Isso é resultado. O
> que seria perda de tempo é apagar este ciclo do notebook e apresentar só os gráficos que
> deram certo.

---

## 6. Ciclo 4 — O setor importa?

### PERGUNTA

*Ativos do mesmo setor se comportam de forma parecida?*

Se sim, "setor" é uma variável explicativa útil e a diversificação setorial faz sentido
prático.

In [ ]:
# EXPLORAÇÃO
retornos_diarios = acoes.pivot_table(index="data", columns="ticker", values="retorno_diario")
correlacao_ativos = retornos_diarios.corr()

# Ordenamos os ativos por setor para que o padrão, se existir, fique visível em blocos
ordem = empresas.sort_values(["setor", "ticker"])["ticker"].tolist()
correlacao_ordenada = correlacao_ativos.loc[ordem, ordem]

fig, ax = plt.subplots(figsize=(8.5, 7))
sns.heatmap(correlacao_ordenada, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlação dos retornos diários — ativos agrupados por setor", pad=14)
ax.set_xlabel("")
ax.set_ylabel("")
plt.show()

print("Setores:")
print(empresas.sort_values("setor")[["ticker", "setor"]].to_string(index=False))

In [ ]:
# Medindo o que o gráfico sugere: correlação média dentro e fora do mesmo setor
setor_por_ticker = empresas.set_index("ticker")["setor"].to_dict()

mesmo_setor, setores_diferentes = [], []
tickers = correlacao_ativos.columns

for i, ativo_a in enumerate(tickers):
    for ativo_b in tickers[i + 1:]:
        valor = correlacao_ativos.loc[ativo_a, ativo_b]
        if setor_por_ticker[ativo_a] == setor_por_ticker[ativo_b]:
            mesmo_setor.append(valor)
        else:
            setores_diferentes.append(valor)

print(f"Pares do MESMO setor      ({len(mesmo_setor):>2}): correlação média {np.mean(mesmo_setor):.3f}")
print(f"Pares de setores DIFERENTES ({len(setores_diferentes):>2}): correlação média {np.mean(setores_diferentes):.3f}")

### DESCOBERTA

Pares do mesmo setor são visivelmente mais correlacionados que pares de setores
diferentes. O bloco dos três financeiros (B3SA3, BBDC4, ITUB4) é o mais evidente do mapa.

Um cuidado necessário: no nosso recorte, "mesmo setor" tem **apenas 3 pares** — todos
financeiros. Ou seja, o que medimos não é "o efeito setor" em geral; é "o efeito de ser
banco". Nada garante que dois ativos de bens industriais se comportariam igual.

### HIPÓTESE

Ativos do mesmo setor compartilham fatores de risco (regulação, ciclo de crédito,
concorrência), o que explicaria a correlação mais alta. Verificar isso a sério exige
vários ativos por setor — o que esta amostra não tem.

---

## 7. Síntese

Toda EDA termina com um resumo do que ficou de pé. Escreva-o **como se fosse para outra
pessoa** — porque será, mesmo que essa pessoa seja você daqui a seis meses.

### O que encontramos

1. **Dispersão altíssima entre ativos.** No mesmo período e no mesmo país, o melhor
   papel multiplicou o capital e o pior destruiu mais de 90% dele. Falar de "a bolsa"
   como um bloco esconde isso.
2. **Risco não explicou retorno nesta amostra** — e a correlação negativa que apareceu à
   primeira vista era um artefato de um único ponto.
3. **A queda da MGLU3 foi um processo de vários anos** (−71% em 2021, −59% em 2022,
   −66% em 2024), e não um choque isolado. A explicação por juros é plausível, mas
   **não encontramos evidência dela nesta amostra**.
4. **Ativos do mesmo setor andam mais juntos**, o que dá base empírica à diversificação
   setorial.

### O que NÃO podemos afirmar

- Nada sobre **causalidade**: em nenhum momento isolamos causa de efeito.
- Nada sobre o **futuro**: descrevemos 2021–2025 e só.
- Nada sobre **"a B3"**: são 8 ações grandes e sobreviventes, não uma amostra do mercado.
- Nada sobre **estratégia de investimento**: descrição do passado não é recomendação.

### O que faríamos a seguir

- ampliar a amostra para todo o Ibovespa, e incluir empresas que saíram da bolsa;
- usar variação não antecipada de juros em vez do nível;
- controlar por tamanho da empresa e por endividamento;
- estender o período para incluir outros ciclos de juros.

> Repare que a síntese tem **três partes**, e as duas últimas são tão importantes quanto
> a primeira. Uma análise que só lista descobertas é propaganda, não análise.

## 8. Checklist de EDA

Um roteiro para você aplicar em qualquer base nova — no TCC, no estágio, em qualquer
análise sua.

```
ANTES DE COMEÇAR
  [ ] Qual é a pergunta? Escreva em uma frase.
  [ ] De onde vieram os dados? Quem os coletou, quando, com que método?
  [ ] O que cada coluna significa? Em que unidade?
  [ ] Que limitações já são conhecidas? Escreva ANTES de olhar os resultados.

ENTENDER A BASE
  [ ] shape · head · tail · info · describe
  [ ] Tipos corretos? Faltantes? Duplicatas?
  [ ] value_counts() em cada coluna categórica
  [ ] Mínimos e máximos fazem sentido no mundo real?

UMA VARIÁVEL DE CADA VEZ
  [ ] Numéricas: histograma, média × mediana, dispersão, assimetria
  [ ] Categóricas: contagem e proporção por categoria
  [ ] Extremos: são erros ou são dados?

DUAS VARIÁVEIS
  [ ] Numérica × numérica: dispersão, depois correlação (nessa ordem)
  [ ] Numérica × categórica: boxplot, comparação de medianas
  [ ] Categórica × categórica: tabela cruzada
  [ ] Sempre: qual o tamanho de cada grupo?

FECHAR
  [ ] O que descobri? (com números)
  [ ] O que NÃO posso afirmar?
  [ ] O que investigaria a seguir?
  [ ] O notebook roda do zero, de cima para baixo, sem erro?
```

## 9. Sete armadilhas da EDA

1. **Pescaria.** Testar tudo contra tudo até algo dar significativo. Com dados
   suficientes, sempre dá.
2. **Confirmar o que já se acreditava.** Você para de explorar quando encontra o
   resultado que queria — e não quando esgotou a pergunta.
3. **Confundir correlação com causalidade.** Já falamos. Vale repetir.
4. **Ignorar o tamanho dos grupos.** Uma média de 3 observações vira barra do mesmo
   tamanho de uma média de 3.000.
5. **Esconder o que não funcionou.** O caminho errado é informação sobre os dados.
6. **Não conferir a qualidade dos dados antes.** Nenhum gráfico bonito conserta uma
   coluna com números salvos como texto.
7. **Apresentar descrição como previsão.** "Rendeu 30% ao ano" não é "vai render 30% ao
   ano".

## 10. Recapitulando

- EDA é um **ciclo**: pergunta → exploração → descoberta → hipótese → nova pergunta.
- Comece **conhecendo os dados** e **declarando as limitações**, antes de olhar
  resultados.
- Cada ciclo começa por uma pergunta escrita. Sem pergunta, não é exploração — é
  passeio.
- **Teste a robustez das descobertas**: remova um ponto, mude o recorte, refaça o
  cálculo de outro jeito. O que sobrevive é resultado; o que não sobrevive era artefato.
- Descobertas sobre a **sua própria análise** ("o cruzamento que fiz não é o certo")
  valem tanto quanto descobertas sobre os dados.
- Termine com três listas: **o que encontrei**, **o que não posso afirmar**, **o que
  faria a seguir**.

**Fim do módulo 03.** Faça a `lista_03_visualizacao.ipynb`: é nela que você percorre
este ciclo por conta própria.